In [ ]:
import datetime, pytz, holidays
from numpy.lib.type_check import nan_to_num
from numpy.core.numeric import NaN
from time import tzname
import pandas as pd
import os

In [ ]:
df_dates = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/df_dates.pkl')
df_dates.index = pd.to_datetime(df_dates.index)

df_dates.Time = pd.to_datetime(df_dates.Time)
df_dates.sort_values(by="Time",inplace=True)
df_dates.index = df_dates.Time

df_dates["Minute"] = df_dates.Time.to_period(freq='T').index


In [ ]:
df_dates

In [ ]:
minutes = df_dates.Minute.unique()
# del df
df_minutes = pd.DataFrame(index =minutes, columns=["Minute", "Date", "LastTime", "Articles", "Title", "Text", "PressReleaseCount", "Company"])

df_minutes.Minutes = minutes
df_minutes.Articles = ""
df_minutes.Title = ""
df_minutes.Text = ""
df_minutes.PressReleaseCount = 0
df_minutes.Company = ""

print(df_minutes)

In [ ]:
total=0
for i, row in df_dates.T.iteritems(): 
    # print(hour)
    minute = row["Minute"]  
    time = row["Time"]
    date = row["Date"]
    #print(date)
    articles = df_minutes.at[minute, "Articles"]
    titles = df_minutes.at[minute, "Title"]
    texts = df_minutes.at[minute, "Text"]
    company = df_minutes.at[minute, "Company"]
    articles

    df_minutes.at[minute, "Articles"] = articles + ". " + row["Articles"]
    df_minutes.at[minute, "Title"] = titles + ". " + row["Title"]
    df_minutes.at[minute, "Text"] = texts + ". " + row["Text"]
    df_minutes.at[minute, "Company"] = company + ". " + row["Company"]
    df_minutes.at[minute, "PressReleaseCount"] +=1
    df_minutes.at[minute, "LastTime"] = time
    df_minutes.at[minute, "Date"] = date
    df_minutes.at[minute, "Minute"] = minute
   
    total +=1

print(total)

In [ ]:
df_minutes = df_minutes["2020-06-01":"2022-06-01"]

# Market Cap

In [ ]:
df_minutes["MarketCap"]=""
df_minutes["MarketCapFactor"]=""

In [ ]:
df_companies=pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SPXCompaniesByMarketCap.pkl")

In [ ]:
def MarketCapFactor(companies,date):
    marketcap = 0
    for company in companies:
        try:
            # print(df_companies.loc[company, date])
            marketcap += df_companies.loc[company, date].values[0]
        except:
            print("Exception: MarketCapFactor")
            # print(company)
            # print(date)
            # print("===============")
    meanMarketCap = df_companies.loc[df_companies.index.get_level_values("date")==date].values.sum()
    MarketCap_Factor = (marketcap/meanMarketCap)
    return MarketCap_Factor

In [ ]:
df_minutes

In [ ]:
for minute, row in df_minutes.T.iteritems():
    marketcap = 0
    date=row["Date"]
    companieslist = list(filter(None, row['Company'].split(". ")))
    df_companies.loc[df_companies.index.get_level_values("date")==date].values.sum()
    for company in companieslist:
        try:
            marketcap += df_companies.loc[company, date].values[0]
        except:
            print(company)
            print(date)
    df_minutes.loc[minute,'MarketCap'] = marketcap
    MarketCap_factor = MarketCapFactor(companieslist, date)
    df_minutes.loc[minute,"MarketCapFactor"] = MarketCap_factor

In [ ]:
df_minutes

In [ ]:
df_minutes.to_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/minuteAggregation.pkl')